In [4]:
## Dataset ## 
import os
import torch
from torchvision import datasets, transforms
from PIL import Image
import numpy as np
import torchvision.transforms.functional as F

class Binarize:
    def __init__(self, threshold):
        self.threshold = threshold

    def __call__(self, img):
        img = img.convert("L")                                           # PIL 이미지를 Grayscale로 변환
        img_np = np.array(img)                                           # NumPy 배열로 변환
        img_bin = (img_np > self.threshold).astype(np.uint8) * 255 
        return Image.fromarray(img_bin)                                  # 다시 PIL 이미지로 변환

In [ ]:
size_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    Binarize(threshold=128),     
    transforms.ToTensor(),  
])

trainset = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Train',  transform = size_transform) 
testset = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Test',  transform = size_transform) 
trainloader = torch.utils.data.DataLoader(trainset, batch_size=20, shuffle=True, num_workers=2, drop_last=True)
testloader = torch.utils.data.DataLoader(testset, batch_size = 20, shuffle=True, num_workers=2, drop_last=True)

In [6]:
import numpy as np 
import torch.optim as optim    
from tqdm import tqdm 

def train(model, device, trainloader, optimizer, criterion, num_epochs):
    history = np.zeros((0, 3))  

    for epoch in tqdm(range(num_epochs)):
        model.train()
        epoch_loss = 0
        correct = 0
        total = 0

        for X, y in trainloader: 
            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            predict = model(X)
            loss = criterion(predict, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            pred_class = predict.argmax(dim=1)
            correct += (pred_class == y).sum().item()
            total += y.size(0)

        avg_loss = epoch_loss / len(trainloader)
        avg_accuracy = correct / total

        item = np.array([epoch + 1, avg_loss, avg_accuracy])
        history = np.vstack((history, item))

        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.6f}, Accuracy: {avg_accuracy:.4f}")

    return history
    
def test(model, device, test_loader, criterion):
    test_loss = []
    test_accuracy = []
    model.eval()

    with torch.no_grad():
        for X, y in tqdm(test_loader): 
            X = X.to(device)
            y = y.to(device)

            predict = model(X)
            loss = criterion(predict, y)

            pred_class = predict.argmax(dim=1)
            accuracy = (pred_class == y).float().mean()

            test_accuracy.append(accuracy.item())
            test_loss.append(loss.item())

    avg_loss = sum(test_loss) / len(test_loss)
    avg_accuracy = sum(test_accuracy) / len(test_accuracy)
    print(f'test loss : {avg_loss:.4f} / test_accuracy : {avg_accuracy:.4f}')

In [7]:
## resnet model ##
import torchvision.models as models
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
prob1_2 = models.resnet18()
num_ftrs = prob1_2.fc.in_features
prob1_2.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
prob1_2.fc = nn.Sequential(
    nn.Linear(num_ftrs, 4),
)

prob1_2 = prob1_2.to(device)

In [16]:
# 하이퍼파라미터 설정
num_epochs = 40
lr = 0.001
optimizer = torch.optim.Adam(prob1_2.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss()

# 학습 실행
history = train(prob1_2, device, trainloader, optimizer, criterion, num_epochs)
result = test(prob1_2, device, testloader, criterion)

  2%|█▊                                                                      | 1/40 [00:10<06:57, 10.71s/it]

Epoch [1/40] - Loss: 1.117552, Accuracy: 0.5028


  5%|███▌                                                                    | 2/40 [00:21<06:47, 10.72s/it]

Epoch [2/40] - Loss: 0.954442, Accuracy: 0.5836


  8%|█████▍                                                                  | 3/40 [00:32<06:35, 10.69s/it]

...

Epoch [38/40] - Loss: 0.037679, Accuracy: 0.9882


 98%|█████████████████████████████████████████████████████████████████████▏ | 39/40 [06:56<00:10, 10.69s/it]

Epoch [39/40] - Loss: 0.021706, Accuracy: 0.9910


100%|███████████████████████████████████████████████████████████████████████| 40/40 [07:06<00:00, 10.67s/it]

Epoch [40/40] - Loss: 0.017043, Accuracy: 0.9951
